In [11]:
%load_ext autoreload
%autoreload 2

import requests
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
%%script false --no-raise-error

url = ("https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json")
response = requests.get(url)

if(response.status_code == 200):
    competitions_data = response.json()
    df_competitions = pd.DataFrame(competitions_data)
    serie_a_2015_16 = df_competitions[
        (df_competitions["competition_name"] == "Serie A")
        & (df_competitions["season_name"] == "2015/2016")
    ]
    
    comp_id = serie_a_2015_16["competition_id"].values[0]
    season_id = serie_a_2015_16["season_id"].values[0]

    print(f"Competition ID: {comp_id}, Season ID: {season_id}")

    url_matches = f"https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/{comp_id}/{season_id}.json"
    response_matches = requests.get(url_matches) 
    
    if response_matches.status_code == 200:
        df_matches = pd.DataFrame(response_matches.json())
        print(f"Số trận đấu: {len(df_matches)}") 
      
else: print("load data fail")

Couldn't find program: 'false'


In [13]:
#check out match data
BASE_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"
MATCH_ID = 3879608  # Verona–Bologna

event_url = f"{BASE_URL}/events/{MATCH_ID}.json"
events_response = requests.get(event_url)
events_response.raise_for_status()
events = events_response.json()

lineup_url = f"{BASE_URL}/lineups/{MATCH_ID}.json"
lineup_response = requests.get(lineup_url)
lineup_response.raise_for_status()
lineups = lineup_response.json()

len(events)

3733

In [14]:
from collections import Counter

type_names = []
event_keys = []

for event in events:
    type_names.append(event["type"]["name"])
    for key in event.keys():
        event_keys.append(key)

#another way
type_names = [event["type"]["name"] for event in events]

#get a list, count the amout of appearances of each element
type_count = Counter(type_names)

# type_count.most_common(15)

# event_count = Counter(event_keys)
# event_count.most_common(15)



In [15]:
import json

shot_event = [] #first shot event

for event in events:
    if (event['type']['name'] == 'Shot'):
        shot_event = event
        break

#print(json.dumps(shot_event, indent=2))  

pass_event = [event for event in events if event['type']['name'] == 'Pass'] #pass events; get all elements
len(pass_event)

sub_event = next(event for event in events if event['type']['name'] == 'Substitution') #get the first true element
sub_event['type']

#get all 3 in 1 loop
type_names = ["Shot", "Pass", "Substitution"]

for type_name in type_names:
    print(f"==={type_name}===")
    event = next(event for event in events if event['type']['name'] == type_name)
    # print(json.dumps(event, indent=2, ensure_ascii=False))


===Shot===
===Pass===
===Substitution===


In [16]:
import sys
from pathlib import Path

#When you run import moneyball, Python looks for it in the folders listed 
#in sys.path. In a notebook, the first of those folders is the 
#notebook's own folder. So we have to set the PROJECT_ROOT inside
#notebook sys.path to import moneyball module
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [22]:
import moneyball.config as config
import moneyball.data_loader as loader

client = loader.StatsBombClient()

events = client.get_events(1)
len(events)


StatsBombError: File is not here